# Node specific parameters with Sklearn training plan

The goal of this notebook is to test tag_parameters functionality with sklearn training plan.

This example uses MNIST dataset. Please check `README.md` file in `notebooks` directory for the instructions to load MNIST dataset and configure nodes.

# Test 1: simple experiment with breakpoints

## Define an experiment model and parameters

In [9]:
from fedbiomed.common.training_plans import FedPerceptron
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.dataset import MnistDataset

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
import numpy as np


class SkLearnClassifierTrainingPlan(FedPerceptron):
    def init_dependencies(self):
        """Define additional dependencies."""
        return ["from sklearn.pipeline import Pipeline",
                "from sklearn.preprocessing import FunctionTransformer",
                "from fedbiomed.common.dataset import MnistDataset",
                "import numpy as np"]
                
    def training_data(self):
        """Prepare data for training.
        
        This function loads a MNIST dataset from the node's filesystem, applies some
        preprocessing and converts the full dataset to a numpy array. 
        Finally, it returns a DataManager created with these numpy arrays.
        """

        pipeline = Pipeline([
                ("norm", FunctionTransformer(
                    lambda im: (np.asarray(im, dtype=np.float64) / 255.0 - 0.1307) / 0.3081,
                    validate=False
                )),
                ("flatten", FunctionTransformer(
                    lambda x: np.ascontiguousarray(x.reshape(-1), dtype=np.float64),
                    validate=False
                )),
            ])

        dataset = MnistDataset(transform=pipeline.transform)
        
        return DataManager(dataset=dataset, shuffle=True)

    def tag_parameters(self, name):
        if name == 'intercept_':
            return {'local', 'persistent'}
        return set()

    


### Model arguments and training arguments

In [10]:
model_args = {'n_features': 28*28,
              'n_classes' : 10,
              'eta0': 1e-6,
              'learning_rate':'constant',
              'alpha':0.1 
              }

training_args = {
    'num_updates': 20, 
    'loader_args': {
        'batch_size': 5,
    },
}

### Create and run the experiment

In [11]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage

tags =  ['#MNIST', '#dataset']
rounds = 15

# select nodes participating in this experiment
exp = Experiment(tags=tags,
                 model_args=model_args,
                 training_plan_class=SkLearnClassifierTrainingPlan,
                 training_args=training_args,
                 round_limit=rounds,
                 save_breakpoints = True,
                 aggregator=FedAverage(),
                 node_selection_strategy=None)

2026-04-15 18:06:26,505 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-04-15 18:06:26,563 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_5eaa634a-8589-4251-9779-f4435535cb17

2026-04-15 18:06:26,564 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_523186bc-f4af-4044-a946-60003cdef368

2026-04-15 18:06:26,567 fedbiomed WARNING - Option share_persistent_buffers is not supported in SKLearnTrainingPlan, it will be ignored.

In [12]:
exp.run()

2026-04-15 18:06:27,794 fedbiomed INFO - Sampled nodes in round 0 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 18:06:27,799 fedbiomed INFO - Sending request 
					 To: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 18:06:27,801 fedbiomed INFO - Sending request 
					 To: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 18:06:27,875 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 1 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss perceptron: 0.000000 
					 ---------

2026-04-15 18:06:27,892 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 1 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss perceptron: 0.000000 
					 ---------

...

2026-04-15 18:06:36,730 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 15 | Iteration: 20/20 (100%) | Samples: 100/100
 					 Loss perceptron: 0.000001 
					 ---------

2026-04-15 18:06:36,736 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 15 | Iteration: 20/20 (100%) | Samples: 100/100
 					 Loss perceptron: 0.000007 
					 ---------

2026-04-15 18:06:37,024 fedbiomed INFO - Nodes that successfully reply in round 14 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 18:06:37,029 fedbiomed INFO - breakpoint number 14 saved at /home/lchambon/Documents/FedBioMed/dev_tests/tests_local_parameters/fbm-components/fbm-researcher/var/experiments/Experiment_0007/breakpoint_0014

15

#### Check model parameters

In [13]:
print("\nList the training rounds : ", exp.aggregated_params().keys())

print("\nAccess the federated params for the last training round :")
print("\t- parameter data: ", exp.aggregated_params()[rounds - 1]['params'].keys())


List the training rounds :  dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14])

Access the federated params for the last training round :
	- parameter data:  dict_keys(['coef_'])


#### Breakpoints

In [14]:
from fedbiomed.researcher.federated_workflows import Experiment
new_exp = Experiment.load_breakpoint('../../fbm-researcher/var/experiments/Experiment_0007/breakpoint_0000')

2026-04-15 18:07:00,670 fedbiomed WARNING - Option share_persistent_buffers is not supported in SKLearnTrainingPlan, it will be ignored.

2026-04-15 18:07:00,673 fedbiomed INFO - Experimentation reload from ../../fbm-researcher/var/experiments/Experiment_0007/breakpoint_0000 successful!

In [15]:
new_exp.run_once(increase=True)

2026-04-15 18:07:03,330 fedbiomed INFO - Sampled nodes in round 1 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 18:07:03,335 fedbiomed INFO - Sending request 
					 To: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 18:07:03,336 fedbiomed INFO - Sending request 
					 To: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-04-15 18:07:03,513 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 2 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss perceptron: 0.000008 
					 ---------

2026-04-15 18:07:03,526 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 2 | Iteration: 1/20 (5%) | Samples: 5/100
 					 Loss perceptron: 0.000009 
					 ---------

...

2026-04-15 18:07:03,743 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_5eaa634a-8589-4251-9779-f4435535cb17 
					 Node Name: Default Node Alias 
					 Round 2 | Iteration: 20/20 (100%) | Samples: 100/100
 					 Loss perceptron: 0.000001 
					 ---------

2026-04-15 18:07:03,766 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_523186bc-f4af-4044-a946-60003cdef368 
					 Node Name: Default Node Alias 
					 Round 2 | Iteration: 20/20 (100%) | Samples: 100/100
 					 Loss perceptron: 0.000011 
					 ---------

2026-04-15 18:07:04,026 fedbiomed INFO - Nodes that successfully reply in round 1 ['NODE_5eaa634a-8589-4251-9779-f4435535cb17', 'NODE_523186bc-f4af-4044-a946-60003cdef368']

2026-04-15 18:07:04,029 fedbiomed INFO - breakpoint number 1 saved at /home/lchambon/Documents/FedBioMed/dev_tests/tests_local_parameters/fbm-components/fbm-researcher/var/experiments/Experiment_0007/breakpoint_0001

1